# 05 — Modélisation : Logistic Regression

**Objectif** : entraîner et optimiser une régression logistique pour la détection d'anomalies CBC, sur les deux configurations de features définies à l'étape 4.

**Rappel des deux expériences :**
- **Expérience A** : 9 features (toutes les variables CBC) → sanity check, data leakage total attendu
- **Expérience B1** : 5 features (HK, MCV, MCHC, MCH, RDW) → problème de prédiction réaliste

**Étapes de ce notebook :**
1. Chargement des données préparées
2. Baseline (paramètres par défaut)
3. Recherche d'hyperparamètres (GridSearchCV)
4. Évaluation finale sur le test set
5. Interprétation des coefficients
6. Sauvegarde des modèles et résultats

In [1]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import time
import joblib
import os

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from evaluation import evaluate_model, plot_confusion_matrices, results_table

os.makedirs('../models', exist_ok=True)
os.makedirs('../figures', exist_ok=True)

## 0. Chargement des données préparées (issues de l'étape 4)

In [2]:
X_train_A = pd.read_csv('../data/processed/X_train_A.csv')
X_test_A  = pd.read_csv('../data/processed/X_test_A.csv')
y_train_A = pd.read_csv('../data/processed/y_train_A.csv').squeeze()
y_test_A  = pd.read_csv('../data/processed/y_test_A.csv').squeeze()

X_train_B1 = pd.read_csv('../data/processed/X_train_B1.csv')
X_test_B1  = pd.read_csv('../data/processed/X_test_B1.csv')
y_train_B1 = pd.read_csv('../data/processed/y_train_B1.csv').squeeze()
y_test_B1  = pd.read_csv('../data/processed/y_test_B1.csv').squeeze()

print("Expérience A :", X_train_A.shape, X_test_A.shape)
print("Expérience B1:", X_train_B1.shape, X_test_B1.shape)

Expérience A : (419075, 9) (104769, 9)
Expérience B1: (419075, 5) (104769, 5)


## 1. Baseline — Logistic Regression avec paramètres par défaut

Premier résultat de référence, avant tout réglage d'hyperparamètres.

In [3]:
# Baseline Expérience A
log_reg_A_baseline = LogisticRegression(max_iter=1000, random_state=42)
model_A_base, y_pred_A_base, metrics_A_base = evaluate_model(
    log_reg_A_baseline, X_train_A, y_train_A, X_test_A, y_test_A,
    "LogReg baseline - Exp A"
)

# Baseline Expérience B1
log_reg_B1_baseline = LogisticRegression(max_iter=1000, random_state=42)
model_B1_base, y_pred_B1_base, metrics_B1_base = evaluate_model(
    log_reg_B1_baseline, X_train_B1, y_train_B1, X_test_B1, y_test_B1,
    "LogReg baseline - Exp B1"
)

results_table([metrics_A_base, metrics_B1_base])

,Modèle,Accuracy,Precision,Recall,F1-score,Temps (s)
0,LogReg baseline - Exp A,0.839514,0.760327,0.641454,0.695850,2.76
1,LogReg baseline - Exp B1,0.823870,0.735078,0.601301,0.661494,0.38


## 2. Recherche d'hyperparamètres — GridSearchCV (Expérience A)

Grille testée :
- `C` : force de régularisation inverse (0.001 à 100)
- `penalty` : l1 ou l2
- `solver` : liblinear (supporte l1+l2) ou lbfgs (l2 seulement)
- `class_weight` : None ou 'balanced' (pour favoriser le Recall sur la classe Anomalie)

Validation croisée à 5 folds, optimisée sur le F1-score (pertinent pour un contexte médical avec classes légèrement déséquilibrées).

In [ ]:
param_grid_logreg = [
    {'penalty': ['l1', 'l2'], 'C': [0.001, 0.01, 0.1, 1, 10, 100],
     'solver': ['liblinear'], 'class_weight': [None, 'balanced']},
    {'penalty': ['l2'], 'C': [0.001, 0.01, 0.1, 1, 10, 100],
     'solver': ['lbfgs'], 'class_weight': [None, 'balanced']}
]

grid_logreg_A = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000, random_state=42),
    param_grid=param_grid_logreg,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=1
)

start = time.time()
grid_logreg_A.fit(X_train_A, y_train_A)
print(f"Temps de recherche: {time.time()-start:.1f}s")

print("\nMeilleurs paramètres (Exp A):", grid_logreg_A.best_params_)
print("Meilleur F1-score (CV, Exp A):", grid_logreg_A.best_score_)

## 3. Recherche d'hyperparamètres — GridSearchCV (Expérience B1)

In [ ]:
grid_logreg_B1 = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000, random_state=42),
    param_grid=param_grid_logreg,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=1
)

start = time.time()
grid_logreg_B1.fit(X_train_B1, y_train_B1)
print(f"Temps de recherche: {time.time()-start:.1f}s")

print("\nMeilleurs paramètres (Exp B1):", grid_logreg_B1.best_params_)
print("Meilleur F1-score (CV, Exp B1):", grid_logreg_B1.best_score_)

## 4. Évaluation finale des meilleurs modèles sur le TEST set

In [ ]:
best_logreg_A = grid_logreg_A.best_estimator_
best_logreg_B1 = grid_logreg_B1.best_estimator_

y_pred_A = best_logreg_A.predict(X_test_A)
y_pred_B1 = best_logreg_B1.predict(X_test_B1)

results = []
for name, y_true, y_pred in [
    ("LogReg baseline - Exp A", y_test_A, y_pred_A_base),
    ("LogReg optimisé - Exp A", y_test_A, y_pred_A),
    ("LogReg baseline - Exp B1", y_test_B1, y_pred_B1_base),
    ("LogReg optimisé - Exp B1", y_test_B1, y_pred_B1),
]:
    results.append({
        'Modèle': name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-score': f1_score(y_true, y_pred),
    })

df_results = results_table(results)
df_results

## 5. Matrices de confusion (baseline vs optimisé, A vs B1)

In [ ]:
configs = [
    (y_test_A, y_pred_A_base, "LogReg baseline - Exp A"),
    (y_test_A, y_pred_A, "LogReg optimisé - Exp A"),
    (y_test_B1, y_pred_B1_base, "LogReg baseline - Exp B1"),
    (y_test_B1, y_pred_B1, "LogReg optimisé - Exp B1"),
]

plot_confusion_matrices(configs, save_path='../figures/confusion_matrices_logreg.png')

## 6. Interprétation — importance des variables (coefficients)

Un coefficient positif augmente la probabilité d'"Anomalie", un coefficient négatif la diminue. Plus la valeur absolue est grande, plus la variable pèse dans la décision du modèle (les features étant normalisées, les coefficients sont directement comparables entre eux).

In [ ]:
coef_A = pd.DataFrame({
    'Variable': X_train_A.columns,
    'Coefficient': best_logreg_A.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

coef_B1 = pd.DataFrame({
    'Variable': X_train_B1.columns,
    'Coefficient': best_logreg_B1.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

print("Importance des variables (Expérience A):")
print(coef_A)
print("\nImportance des variables (Expérience B1):")
print(coef_B1)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].barh(coef_A['Variable'], coef_A['Coefficient'], color='steelblue')
axes[0].set_title('Coefficients - Expérience A')
axes[0].axvline(0, color='black', linewidth=0.8)

axes[1].barh(coef_B1['Variable'], coef_B1['Coefficient'], color='salmon')
axes[1].set_title('Coefficients - Expérience B1')
axes[1].axvline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.savefig('../figures/coefficients_logreg.png', dpi=150)
plt.show()

## 7. Sauvegarde des modèles et résultats

In [ ]:
joblib.dump(best_logreg_A, '../models/logreg_A_best.pkl')
joblib.dump(best_logreg_B1, '../models/logreg_B1_best.pkl')

df_results.to_csv('../data/processed/results_logreg.csv', index=False)
print("✅ Modèles et résultats Logistic Regression sauvegardés")

## Ce qu'il faut analyser pour le rapport

1. **Baseline vs optimisé** : le GridSearch a-t-il significativement amélioré le F1-score, ou les paramètres par défaut étaient déjà proches de l'optimal ?
2. **Expérience A vs B1** : un écart important de performance entre les deux confirme visuellement l'effet du data leakage anticipé à l'étape 4.
3. **`class_weight='balanced'`** : a-t-il été retenu par le GridSearch ? Si oui, quel est son effet sur le couple Precision/Recall ?
4. **Coefficients** : les variables à poids fort dans le score médical pondéré (ERY, LEUKO, HB, PLT) ressortent-elles aussi comme les plus influentes dans le modèle (Expérience A) ?